# Data Preprocessing for Machine Learning

**Learning objectives:** By the end of this notebook, you will be able to:
- Select and drop irrelevant features from a dataset
- Slice, engineer, and encode new features
- Normalize and standardize numerical features correctly
- Understand *why* each step matters in a real ML pipeline


> In the industry, data scientists spend **60–80% of their time on data preprocessing**. Interviewers love asking about it because messy, real-world data is the norm — not the exception. Mastering this notebook means you'll have solid answers to questions like *"Walk me through how you prepare data for a model."*

---

### Dataset used: Medical Cost Personal Datasets

Features/Columns

- age: age of primary beneficiary
- sex: insurance contractor gender, female, male
- bmi: Body mass index, providing an understanding of body, weights that are relatively high or low relative to height, objective index of body weight (kg / m ^ 2) using the ratio of height to weight, ideally 18.5 to 24.9
- children: Number of children covered by health insurance / Number of dependents
- smoker: Smoking

- region: the beneficiary's residential area in the US, northeast, southeast, southwest, northwest.
- blood pressure
- has_health_insurance: Yes/No
- enrollment_date: when they signed up for insurance

Target:
- charges: Individual medical costs billed by health insurance


## 0. Imports and Setup

> 🏢 **Professional practice:** Group all your imports at the top of the notebook. Use a dedicated cell, add a comment per group (standard library / third-party / local), and pin library versions. This makes collaboration and reproducibility much easier.

In [1]:
# Standard library
import warnings

# Data manipulation & visualisation
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline

# Display settings
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.3f}".format)
sns.set_theme(style="whitegrid")

# Reproducibility — always set a random seed!
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("✅ Setup complete")

✅ Setup complete


## 1. Load and Explore the Data

> 🏢 **Professional practice:** Never jump straight into transformations. Always start with **Exploratory Data Analysis (EDA)** — understand the shape, types, missing values, and distributions first. Skipping this step is a common mistake that leads to bugs later.

In [2]:
filename = "./data/insurance.csv"
df = pd.read_csv(filename)

print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns\n")
df.head()

Shape: 1338 rows × 11 columns



,patient_id,age,sex,bmi,children,smoker,region,charges,blood_pressure,has_health_insurance,enrollment_date
0,0,19,female,27.900,0,yes,southwest,16884.924,127.451,Yes,2023-06-08
1,1,18,male,33.770,1,no,southeast,1725.552,NaN,Yes,2023-01-14
2,2,28,male,33.000,3,no,southeast,4449.462,129.715,Yes,2024-01-08
3,3,33,male,22.705,0,no,northwest,21984.471,142.845,Yes,2023-08-13
4,4,32,male,28.880,0,no,northwest,3866.855,116.488,Yes,2024-05-25


In [38]:
# Quick type + missing-value overview
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   patient_id            1338 non-null   int64  
 1   age                   1338 non-null   int64  
 2   sex                   1338 non-null   object 
 3   bmi                   1338 non-null   float64
 4   children              1338 non-null   int64  
 5   smoker                1338 non-null   object 
 6   region                1338 non-null   object 
 7   charges               1338 non-null   float64
 8   blood_pressure        1128 non-null   float64
 9   has_health_insurance  1338 non-null   object 
 10  enrollment_date       1338 non-null   object 
dtypes: float64(3), int64(3), object(5)
memory usage: 115.1+ KB


## 2. Feature Selection — Dropping Useless Columns

Some columns don't add predictive power to a model:
- **Identifiers**: unique per row → the model would just memorise them
- **Near-constant columns**: if 99% of values are the same, there's little information
- **Leaky columns**: columns that would only be known *after* prediction time

> 🏢 **Professional practice:** Document *why* you're dropping a column — your future self and teammates will thank you. Use comments or a dedicated config dictionary.

In [39]:
# Columns to drop and the reason for each
COLS_TO_DROP = {
    "patient_id": "Unique identifier — no predictive power",
    "has_health_insurance": "Almost all patients have health insurance",
}

# Drop only columns that exist (safe for re-runs)
cols_present = [c for c in COLS_TO_DROP if c in df.columns]
df = df.drop(columns=cols_present)

print(f"Dropped {len(cols_present)} column(s): {cols_present}")
print(f"Remaining shape: {df.shape}")

Dropped 2 column(s): ['patient_id', 'has_health_insurance']
Remaining shape: (1338, 9)


## 3. Feature Slicing — Extracting Information from Dates

A raw date string like `"10132018"` is useless to a model. But the *day of the week* or *month* might be very informative. This is called **feature slicing** — extracting smaller, more useful features from a single column.

> 🏢 **Professional practice:** Use `pd.to_datetime()` instead of string manipulation. It is safer, faster, and exposes many useful accessors (`.dt.month`, `.dt.dayofweek`, etc.).

In [4]:
# Inspect the raw format first
print("Sample values:", df["enrollment_date"].head().tolist())
print("Dtype:", df["enrollment_date"].dtype)

Sample values: ['2023-06-08', '2023-01-14', '2024-01-08', '2023-08-13', '2024-05-25']
Dtype: object


### ✏️ Exercise  — Engineer a new feature 

The enrollment_date column is currently stored as a text string (e.g., "2024-11-14"). A machine learning model cannot natively read this.

- Convert the column into a proper pandas datetime object.
- "Slice" the column to extract three new numerical columns: enrollment_year, enrollment_month, and enrollment_day_of_week (where 0 = Monday, 6 = Sunday).
- Once the pieces are extracted, drop the original text enrollment_date column.

In [ ]:
# YOUR CODE HERE

,enrollment_date,day,month,year,day_of_week
0,2023-06-08,8,6,2023,3
1,2023-01-14,14,1,2023,5
2,2024-01-08,8,1,2024,0
3,2023-08-13,13,8,2023,6
4,2024-05-25,25,5,2024,5


*Your interpretation here (double-click to edit):*


## 4. Feature Engineering — Creating New Features

Feature engineering means *creating* new columns that might be more useful than the originals. This is often where domain knowledge has the biggest impact.

### ✏️ Exercise  — Engineer a new feature 

Create a new binary interaction column called high_risk_smoker.

This column should equal 1 if the patient is a smoker and has a BMI greater than 30 (the clinical threshold for obesity). Otherwise, it should equal 0

In [ ]:
# YOUR CODE HERE

## 5. Encoding Categorical Features

Machine learning models only understand numbers. We need to convert string categories into numerical representations. There are two main approaches:

| Method | When to use | Output |
|---|---|---|
| **One-Hot Encoding** | No ordinal relationship between categories | Multiple 0/1 columns |
| **Label / Ordinal Encoding** | There is a natural order (e.g. small < medium < large) | A single integer column |


> ⚠️ **Common mistake:** Using Label Encoding on a nominal variable (e.g. colour: red=0, blue=1, green=2) implies a numerical order that doesn't exist. This can mislead the model. Use One-Hot Encoding instead.

> 🏢 **Professional practice:** Use scikit-learn's `OneHotEncoder` (not just `pd.get_dummies`) when building ML pipelines. It can be fitted on train data and then applied consistently to test data — which `pd.get_dummies` cannot.

### 5a. Quick approach with `pd.get_dummies` (for EDA)

Useful when exploring data. **Do not use this in a production ML pipeline.**

In [7]:
# drop_first=True avoids multicollinearity: if smoker_no=0, we already know smoker_yes=1
smoker_dummies = pd.get_dummies(df["smoker"], prefix="smoker", drop_first=True)
smoker_dummies = smoker_dummies.rename(columns={"smoker_yes": "smoker_binary"})
smoker_dummies.head()

,smoker_binary
0,True
1,False
2,False
3,False
4,False


### 5b. Pipeline approach with `sklearn.OneHotEncoder` (for ML pipelines)

The key advantage: you **fit** on training data and **transform** both train and test consistently. Explore sklearn's documentation for OneHotEncoder then apply it to the reaction column. 

In [14]:
# YOUR CODE HERE
# Fit the encoder on the smoker column
encoder = OneHotEncoder(drop="first")
# Transform the existing data
smoker_encoded = encoder.fit_transform(df[["smoker"]])

# Simulate new, unseen data arriving — the encoder handles it consistently
smoker_new_data = pd.DataFrame({"smoker": ["yes", "no", "yes"]})

# Apply the same transformation to the new data
smoker_new_encoded = encoder.transform(smoker_new_data[["smoker"]])
print(smoker_new_encoded.toarray())

[[1.]
 [0.]
 [1.]]


Why should you not use get_dummies when training a machine learning model? What are other types of Encoders available through sklearn?

### ✏️ Exercise  — Encode `Primary Fur Color` (5 min)

1. Print the unique values in `Primary Fur Color`.
2. Is this a nominal or ordinal variable? Should you use One-Hot Encoding or Label Encoding? Justify your choice.
3. Apply your chosen encoding using `OneHotEncoder` from sklearn. Print the result.

In [ ]:
# YOUR CODE HERE



## 6. Handling Missing Values

> 🏢 **Professional practice:** Never silently ignore missing values — always have an explicit strategy and document it. Dropping rows/columns is a last resort. Understand *why* values are missing before deciding how to handle them.

When preparing data for an ML model, we want to preserve as many observations as possible. Three common strategies:
- **Drop** rows or columns (only if missing % is very high or missing is meaningful)
- **Fill with a statistic** (mean, median for numeric; mode for categorical)
- **Predict** missing values using a model (advanced)

In [ ]:
# Visualise missing values


In [ ]:
# Strategy: fill categorical columns with "Unknown" (explicit, auditable) 
# (Hint: use df.select_dtypes to select only categorical columns)


# Strategy: fill numerical columns with the median (robust to outliers)

# Check that all missing values have been handled
print(f"Missing values remaining: {df.isnull().sum().sum()}")

## 7. Feature Scaling — Normalisation and Standardisation

Many ML algorithms are sensitive to the scale of features. For example:
- A column ranging `0–1` and another ranging `0–100,000` would give the large-range column unfair dominance.
- Algorithms like k-NN, SVM, and neural networks are especially affected.
- Tree-based models (Random Forest, XGBoost) are generally **not** affected.

### Two main approaches:

| Method | Formula | When to use |
|---|---|---|
| **Min-Max Scaling** (Normalisation) | $z = \frac{x - \min}{\max - \min}$ → range [0, 1] | When you need bounded output; no strong outliers |
| **Z-score Standardisation** | $z = \frac{x - \mu}{\sigma}$ → mean=0, std=1 | Most general-purpose choice; handles outliers better |

> ⚠️ **Critical rule: always fit on training data only, then transform both train and test.**  
> Fitting on all data *before* splitting is called **data leakage** — your model secretly "sees" the test set statistics during training, making evaluation unreliable. This is one of the most common and costly mistakes in ML.

In [ ]:
# --- STEP 1: Split FIRST ---
# YOUR CODE HERE


# Print the shapes of the resulting datasets

In [ ]:
# --- STEP 2: Fit standard scaler on train, transform both ---

# Verify that the training set has been standardized correctly

In [ ]:
# Visualise the effect of scaling on one feature

# Reapeat for MinMaxScaler and compare the results

## 8. Putting It All Together: A `sklearn` Pipeline

> 🏢 **Professional practice:** In real projects, preprocessing steps are assembled into a `Pipeline`. This prevents data leakage, makes code reproducible and clean, and allows you to tune all steps together.

Go to the sklearn documentation and create a simple pipeline to your numeric data with a `StandardScaler()` and categorical data with  `OneHotEncoding`.

In [ ]:
# YOUR CODE HERE

### Additional Resources

**Concepts**
- [Feature Engineering — Towards Data Science](https://towardsdatascience.com/feature-engineering-for-machine-learning-3a5e293a5114)
- [Data Leakage — Kaggle](https://www.kaggle.com/alexisbcook/data-leakage)
- [Missing Data — Towards Data Science](https://towardsdatascience.com/how-to-handle-missing-data-8646b18db0d4)

**scikit-learn docs**
- [Preprocessing overview](https://scikit-learn.org/stable/modules/preprocessing.html)
- [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html)
- [StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html)
- [OneHotEncoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html)